# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
print('hello123')

hello123


In [2]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128 --timeout 10000 --retries 10

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128


In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.12.0.dev20260408+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [3]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade --timeout 10000


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [5]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 학습데이터 200개만 추출
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

c:\YSSAFY\SSAFY\AI II\AI 1차 챌린지\ai2-chellange-data\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0828 12:46:29.364000 28792 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다. <- 구라임 100~200분 소요됨

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [6]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
c:\YSSAFY\SSAFY\AI II\AI 1차 챌린지\ai2-chellange-data\venv\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but

trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [7]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [10]:
train_df = train_df.sample(
    n=200,
    random_state=SEED
).reset_index(drop=True)

# 90% train / 10% valid
split = int(len(train_df) * 0.9)

train_subset = train_df.iloc[:split].reset_index(drop=True)
valid_subset = train_df.iloc[split:].reset_index(drop=True)

print("train:", len(train_subset))
print("valid:", len(valid_subset))

class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a = str(row["a"])
        b = str(row["b"])
        c = str(row["c"])
        d = str(row["d"])

        user_text = build_mc_prompt(q, a, b, c, d)

        prompt_messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": SYSTEM_INSTRUCT
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": img
                    },
                    {
                        "type": "text",
                        "text": user_text
                    }
                ]
            }
        ]

        sample = {
            "prompt_messages": prompt_messages,
            "image": img,
        }

        if self.train:
            gold = str(row["answer"]).strip().lower()

            # 혹시 데이터가 이상하면 바로 잡기
            if gold not in {"a", "b", "c", "d"}:
                raise ValueError(
                    f"Invalid answer at index {i}: {gold}"
                )

            full_messages = prompt_messages + [
                {
                    "role": "assistant",
                    "content": [
                        {
                            "type": "text",
                            "text": gold
                        }
                    ]
                }
            ]

            sample["full_messages"] = full_messages
            sample["gold"] = gold

        return sample


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):

        images = [sample["image"] for sample in batch]

        # =========================
        # inference / test
        # =========================
        if not self.train:
            prompt_texts = []

            for sample in batch:
                prompt_text = self.processor.apply_chat_template(
                    sample["prompt_messages"],
                    tokenize=False,
                    add_generation_prompt=True
                )

                prompt_texts.append(prompt_text)

            enc = self.processor(
                text=prompt_texts,
                images=images,
                padding=True,
                return_tensors="pt"
            )

            return enc

        # =========================
        # training
        # =========================

        full_texts = []
        prompt_texts = []

        for sample in batch:

            # 정답까지 포함
            full_text = self.processor.apply_chat_template(
                sample["full_messages"],
                tokenize=False,
                add_generation_prompt=False
            )

            # assistant 답변 직전까지
            prompt_text = self.processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True
            )

            full_texts.append(full_text)
            prompt_texts.append(prompt_text)

        # -------------------------
        # 전체 sequence encoding
        # -------------------------

        full_enc = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        # -------------------------
        # prompt 길이 측정
        # -------------------------

        prompt_enc = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        labels = full_enc["input_ids"].clone()

        # 기본적으로 모든 token loss 무시
        labels[:] = -100

        # -------------------------
        # assistant answer만 label
        # -------------------------

        for i in range(len(batch)):

            # 실제 full sequence 길이
            full_len = int(
                full_enc["attention_mask"][i].sum().item()
            )

            # assistant 답변 시작 위치
            prompt_len = int(
                prompt_enc["attention_mask"][i].sum().item()
            )

            # 현재 batch_size=1 기준으로 가장 직관적
            labels[i, prompt_len:full_len] = \
                full_enc["input_ids"][i, prompt_len:full_len]

        full_enc["labels"] = labels

        return full_enc

train: 180
valid: 20


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [11]:
train_ds = VQAMCDataset(
    train_subset,
    processor,
    train=True
)

valid_ds = VQAMCDataset(
    valid_subset,
    processor,
    train=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(
        processor=processor,
        train=True
    ),
    num_workers=0
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(
        processor=processor,
        train=True
    ),
    num_workers=0
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [12]:

from tqdm.auto import tqdm
import math
import torch

NUM_EPOCHS = 20
GRAD_ACCUM = 4
LR = 1e-4

# device_map="auto" + 4bit 모델이면
# model = model.to(device) 는 하지 않는 편이 안전함

# LoRA 등 실제 학습되는 파라미터만 optimizer에 전달
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR
)

# epoch당 optimizer step 수
steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

# 전체 optimizer step 수
num_training_steps = (
    NUM_EPOCHS * steps_per_epoch
)

# warmup 3%
num_warmup_steps = int(
    num_training_steps * 0.03
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print("Train batches:", len(train_loader))
print("Steps / epoch:", steps_per_epoch)
print("Total optimizer steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)

Train batches: 180
Steps / epoch: 45
Total optimizer steps: 900
Warmup steps: 27


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [13]:
global_step = 0

# 처음부터 gradient 초기화
optimizer.zero_grad(set_to_none=True)

for epoch in range(NUM_EPOCHS):

    # --------------------
    # Train
    # --------------------
    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]",
        unit="batch"
    )

    for step, batch in enumerate(
        progress_bar,
        start=1
    ):

        # model이 올라가 있는 device 기준
        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        # BF16 사용
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):
            outputs = model(**batch)

            raw_loss = outputs.loss

            # gradient accumulation용
            loss = raw_loss / GRAD_ACCUM

        # BF16에서는 GradScaler 불필요
        loss.backward()

        # 실제 raw loss 기록
        train_loss_sum += raw_loss.item()
        train_batches += 1

        # gradient accumulation 완료
        # 또는 마지막 batch
        should_step = (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        )

        if should_step:

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

        avg_train_loss = (
            train_loss_sum / train_batches
        )

        progress_bar.set_postfix({
            "loss": f"{avg_train_loss:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}"
        })


    # --------------------
    # Validation
    # --------------------
    model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        val_bar = tqdm(
            valid_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]",
            unit="batch"
        )

        for batch in val_bar:

            batch = {
                k: v.to(model.device)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):
                outputs = model(**batch)
                val_loss = outputs.loss

            val_loss_sum += val_loss.item()
            val_batches += 1

            val_bar.set_postfix({
                "loss": f"{val_loss_sum / val_batches:.4f}"
            })

    avg_train_loss = (
        train_loss_sum / max(train_batches, 1)
    )

    avg_val_loss = (
        val_loss_sum / max(val_batches, 1)
    )

    print(
        f"\n[Epoch {epoch+1}/{NUM_EPOCHS}] "
        f"train loss: {avg_train_loss:.4f} | "
        f"valid loss: {avg_val_loss:.4f}"
    )


# ==================== 모델 저장 ====================

SAVE_DIR = "/content/qwen2_5_vl_3b_lora"

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("Saved:", SAVE_DIR)

Epoch 1/20 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]c:\YSSAFY\SSAFY\AI II\AI 1차 챌린지\ai2-chellange-data\venv\Lib\site-packages\torch\utils\checkpoint.py:237: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
c:\YSSAFY\SSAFY\AI II\AI 1차 챌린지\ai2-chellange-data\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
Epoch 1/20 [valid]: 100%|██████████| 20/20 [00:06<00:00,  3.00batch/s, loss=0.3474]



[Epoch 1/20] train loss: 0.1963 | valid loss: 0.3474


Epoch 2/20 [valid]: 100%|██████████| 20/20 [00:06<00:00,  2.95batch/s, loss=0.3645]



[Epoch 2/20] train loss: 0.0800 | valid loss: 0.3645


Epoch 3/20 [valid]: 100%|██████████| 20/20 [00:06<00:00,  3.10batch/s, loss=0.5797]



[Epoch 3/20] train loss: 0.0181 | valid loss: 0.5797


Epoch 4/20 [train]:  29%|██▉       | 52/180 [00:37<01:31,  1.40batch/s, loss=0.0032, lr=8.61e-05]


KeyboardInterrupt: 

In [24]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
이 사진에 보이는 음식 중 치킨 너겟과 함께 제공된 음식은 무엇인가요?
(a) 피클
(b) 감자튀김
(c) 케첩
(d) 셀러리 스틱과 소스

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
d
